### Putting it all together

In [1]:
import sys
import os

# sys.path.append(os.path.abspath('E:\Vault\TheUnknownDimension\TerrestraX')) 
# print (sys.path)

sys.path.append(r"E:\Vault\TheUnknownDimension\TerrestraX")

from core.kinematics import dh_transform, chain_kin, chain_jacobian
from core.dtypes import ChainParams, JointVec, NpSym, Vec3
from robots.rconfig import FL_chain
from sympy import symbols, Matrix
from pprint import pprint

chain = FL_chain()

q = symbols('q:3', real=True)

# print(chain.dh_params[1])
# print(type(chain.dh_params[0]))
# DHtr = dh_transform(chain.dh_params[1], q[0])
# pprint(DHtr.subs(q[0], 0.0))
# pprint(dh_transform(chain.dh_params[1], q[0]).subs(q[0], 0.0))

T0_base = Matrix([
    [1, 0, 0, 0.060000],
    [0, 1, 0, 0.060000],
    [0, 0, 1, 0.094700],
    [0, 0, 0, 1]
])
T = chain_kin(chain, q)

T = T0_base * T

T_eval = T.evalf(subs={q[0]: 0.0, q[1]: 0.0, q[2]: 0.0})

pprint(T_eval)
# print("T_eval shape: ", T_eval.shape)
T_foot = T_eval[:3, 3]
pprint(T_foot)
pprint(T_foot.shape)
# pprint("T_foot shape: ", T_foot.shape)
# Check for 3-4 poses to validate the kinematics

J = chain_jacobian(chain, q)
J_eval = J.evalf(subs={q[0]: 0.0, q[1]: 0.0, q[2]: 0.0})
pprint(J_eval.shape)
# print("J_eval shape:", J_eval.shape)


Matrix([
[                  1.0, -2.28482827799104e-17, -8.63305738164096e-17,    0.165784461165793],
[-2.08310030769952e-17,                   1.0, -9.13779171947447e-17,     0.16578438280712],
[ 3.52233187579518e-17,  3.52232737146378e-17,                   1.0, -4.87332976395059e-5],
[                    0,                     0,                     0,                  1.0]])
Matrix([
[   0.165784461165793],
[    0.16578438280712],
[-4.87332976395059e-5]])
(3, 1)
(3, 3)


In [3]:
# print(Matrix([0.01, 0.0, 0.0]).shape)
del_q = J_eval.pinv() * Matrix([0.1658, 0.1658, 0.0])
pprint(del_q*180/3.14159)

# ik
# trajectory 
# connect to kinematics


Matrix([
[  3.39028578637711e-5],
[-0.000177396405179082],
[     191.922117506339]])


In [5]:
from core.dtypes import IKParams
from core.inverse_kinematics import LegIK
import numpy as np

ik = LegIK(chain, T0_base)
q0 = np.array([0.0, 0.0, 0.0], dtype=float)
target = np.array([0.1658, 0.1658, 0.0], dtype=float)
# target = np.array([0.1642, 0.1674, 0.0], dtype=float)
# target = np.array([0.1845, 0.1256, 0.0], dtype=float)
# target = np.array([0.10, 0.10, 0.02], dtype=float)

q1, info = ik.step_solve(q0, target, IKParams())
pprint(np.round(q1*180/3.14159, 2))
pprint(info)

array([ 0.2 ,  0.01, -0.01])
IKinfo(ok=False, iters=1, err=0.002263429043317829)
